# Gurobi Illustration: Maximize $c_{0,0}M^6/(8\pi G)$

This short notebook reproduces the linear-programming illustration for the gravity-pole bootstrap.  It builds the fixed-`t` gravity-pole constraints, adds crossing null constraints, and uses Gurobi to maximize the leading regular coefficient.

The LP variable order is

\[
(c_{0,0}^{\rm raw}, \rho_{1,0}, \rho_{1,2}, \ldots), \qquad \rho_{p,\ell}\ge 0.
\]

The raw solver normalization uses `GRAVITY_POLE_RHS_SCALE = 1e-4`, so the note-normalized bound is

\[
\frac{c_{0,0}M^6}{8\pi G} = \frac{c_{0,0}^{\rm raw}}{10^{-4}}.
\]


## 1. Import the compact LP helper

The definitions are in `gurobi_c00_bound.py` so that this notebook stays short.


In [ ]:
import csv
import importlib
from pathlib import Path

import gurobi_c00_bound as g0
importlib.reload(g0)


## 2. Choose the numerical resolution

The note setting is

\[
N_\mu=300, \qquad J_{\max}=3000, \qquad N_t=6, \qquad 55\ \text{null constraints}.
\]

This can be slow because it has about `450k` positive spectral variables.  For a quick smoke test, set `USE_NOTE_RESOLUTION = False`.


In [ ]:
USE_NOTE_RESOLUTION = True

if USE_NOTE_RESOLUTION:
    cfg = g0.G0Config(
        n_mu=300,
        j_max=3000,
        n_t=6,
        max_null_power=10,   # gives 55 null constraints
        output_flag=0,
    )
else:
    cfg = g0.G0Config(
        n_mu=80,
        j_max=400,
        n_t=6,
        max_null_power=6,    # gives 21 null constraints
        output_flag=0,
    )

cfg


## 3. Solve the LP with Gurobi

This cell constructs the equality matrix and solves

\[
\max c_{0,0}^{\rm raw}
\]

subject to the fixed-`t` gravity-pole rows, the `k=0` moment row, crossing null constraints, and positivity of the spectral density.


In [ ]:
result = g0.solve_c00_bound(cfg)

summary = {
    key: value
    for key, value in result.items()
    if key not in ('mu_values', 'j_values', 'row_names')
}
summary


## 4. Compare with the saved note table

The saved CSV table was produced by the longer exploratory notebook.  For the note setting, the last row gives approximately

\[
\max c_{0,0}M^6/(8\pi G) \simeq 2.969.
\]


In [ ]:
rows = []
with Path('amxsusy_g0_null_convergence.csv').open(newline='') as f:
    for row in csv.DictReader(f):
        rows.append(row)

saved = [
    row for row in rows
    if int(float(row['num_null'])) == result['num_null_constraints']
]

if saved:
    saved_raw = float(saved[-1]['c00_max'])
    saved_normalized = saved_raw / g0.GRAVITY_POLE_RHS_SCALE
else:
    saved_raw = None
    saved_normalized = None

comparison = {
    'current_c00_over_8piG': result['c00_over_8piG'],
    'saved_c00_over_8piG': saved_normalized,
    'difference': result['c00_over_8piG'] - saved_normalized if saved_normalized is not None else None,
}
comparison


## 5. Save the illustration result

The output CSV is small and records only the compact summary of the LP solve.


In [ ]:
out_path = Path('gurobi_c00_bound_illustration_result.csv')
fieldnames = [
    'ok', 'status_name', 'c00_raw', 'c00_over_8piG',
    'num_variables', 'num_equalities', 'num_null_constraints',
    'n_mu', 'j_max', 'n_t', 'max_null_power',
]
with out_path.open('w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerow({key: result.get(key) for key in fieldnames})

out_path
